# FIRELINE — Kabupaten/Kota Drill-Down

Working notebook for Phase 1 onward. This notebook is the single working surface for validating the new administrative boundaries, normalizing population context, assigning grid-days, producing kabupaten summaries, and preparing dashboard outputs.

The existing grid-day priority score is kept unchanged.

## Working rules

- Primary boundaries come from `data/additions/*.geojson`.
- `kode_kk` is the candidate canonical kabupaten ID.
- Grid-day assignment uses the representative observed point.
- Population is contextual and never enters the priority score.
- Validation results and derived tables should be saved under `review_outputs/`.
- Reusable implementation should be promoted into the existing `fireline/` package only after the notebook logic is validated.

In [1]:
from pathlib import Path
import csv
import hashlib
import json

import geopandas as gpd
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
ADDITIONS = ROOT / 'data' / 'additions'
OUTPUT = ROOT / 'review_outputs'
OUTPUT.mkdir(exist_ok=True)
ADDITIONS

WindowsPath('C:/Users/Inaka/Downloads/Fireline-checkpoint3/Fireline-checkpoint3/data/additions')

## Phase 1 — Boundary inventory

In [2]:
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

boundary_rows = []
boundary_frames = []
for path in sorted(ADDITIONS.glob('*.geojson')):
    frame = gpd.read_file(path).to_crs(4326)
    boundary_frames.append(frame.assign(_source_file=path.name))
    boundary_rows.append({
        'file': path.name,
        'features': len(frame),
        'crs': str(frame.crs),
        'geometry_types': sorted(frame.geometry.geom_type.dropna().unique()),
        'invalid_geometries': int((~frame.geometry.is_valid).sum()),
        'empty_geometries': int(frame.geometry.is_empty.sum()),
        'columns': [c for c in frame.columns if c != 'geometry'],
        'sha256': sha256(path),
    })
boundary_inventory = pd.DataFrame(boundary_rows)
boundary_inventory

,file,features,crs,geometry_types,invalid_geometries,empty_geometries,columns,sha256
0,Provinsi Kalimantan Barat-KAB_KOTA.geojson,14,EPSG:4326,[MultiPolygon],0,0,"[kode_kk, kode_prov, kab_kota, provinsi, fid, ...",281451c2b73fbf5025c56316ad02302a9eb282f3bd33ce...
1,Provinsi Kalimantan Selatan-KAB_KOTA.geojson,13,EPSG:4326,[MultiPolygon],0,0,"[kode_kk, kode_prov, kab_kota, provinsi, fid, ...",5eaf5abd0da69a9d27d2e40ee4ce8454f41626474217fa...
2,Provinsi Kalimantan Tengah-KAB_KOTA.geojson,14,EPSG:4326,[MultiPolygon],0,0,"[kode_kk, kode_prov, kab_kota, provinsi, fid, ...",c42bd23e19bbb8758749b8fe44b66a1545baa8135b066d...
3,Provinsi Kalimantan Timur-KAB_KOTA.geojson,10,EPSG:4326,[MultiPolygon],0,0,"[kode_kk, kode_prov, kab_kota, provinsi, fid, ...",df89c26d7ce054ec1f390d2a52fa050625c853c3ccf45b...
4,Provinsi Kalimantan Utara-KAB_KOTA.geojson,5,EPSG:4326,[MultiPolygon],0,0,"[kode_kk, kode_prov, kab_kota, provinsi, fid, ...",fdc59f685c71bbf7d384030cb895faba423bdac3a3c9dc...


In [3]:
kabupaten = gpd.GeoDataFrame(pd.concat(boundary_frames, ignore_index=True), crs=4326)
required = {'kode_kk', 'kode_prov', 'kab_kota', 'provinsi', 'geometry'}
missing = required - set(kabupaten.columns)
assert not missing, f'Missing boundary fields: {sorted(missing)}'
assert kabupaten.kode_kk.notna().all()
assert kabupaten.kode_kk.astype(str).is_unique
assert kabupaten.geometry.is_valid.all()
kabupaten = kabupaten.rename(columns={
    'kode_kk': 'kabupaten_code',
    'kode_prov': 'province_code',
    'kab_kota': 'kabupaten_name',
    'provinsi': 'province_name',
})
kabupaten[['province_code', 'province_name', 'kabupaten_code', 'kabupaten_name', '_source_file']].head()

,province_code,province_name,kabupaten_code,kabupaten_name,_source_file
0,61,Kalimantan Barat,61.07,Bengkayang,Provinsi Kalimantan Barat-KAB_KOTA.geojson
1,61,Kalimantan Barat,61.02,Mempawah,Provinsi Kalimantan Barat-KAB_KOTA.geojson
2,61,Kalimantan Barat,61.06,Kapuas Hulu,Provinsi Kalimantan Barat-KAB_KOTA.geojson
3,61,Kalimantan Barat,61.71,Kota Pontianak,Provinsi Kalimantan Barat-KAB_KOTA.geojson
4,61,Kalimantan Barat,61.12,Kubu Raya,Provinsi Kalimantan Barat-KAB_KOTA.geojson


## Phase 1 — Population inventory

Population sources have different layouts and units. The next cells should normalize only confirmed kabupaten-level sources. Province totals and unmatched rows remain audit outputs, not silent joins.

In [4]:
population_files = sorted(ADDITIONS.glob('*.csv')) + sorted(ADDITIONS.glob('*.xlsx'))
[(p.name, p.suffix, p.stat().st_size, sha256(p)) for p in population_files]

[('Jumlah Penduduk menurut Kabupaten_Kota di Provinsi Kalimantan Timur, 2024.csv',
  '.csv',
  309,
  '9dbaf2db9ad5fb42634bcec5d208b5b3fe09d6ea37217a4609c031527e7208a1'),
 ('Jumlah Penduduk Menurut Kabupaten_Kota, 2024.csv',
  '.csv',
  328,
  '9df2b10def456a6febe763c60b7cc9df00689d34b280b6d49ceae3e589db91f2'),
 ('Penduduk, Laju Pertumbuhan Penduduk, Distribusi Persentase Penduduk Kepadatan Penduduk, Rasio Jenis Kelamin Penduduk Menurut Kabupaten_Kota di Provinsi Kalimantan Selatan, 2024.csv',
  '.csv',
  1348,
  '9b760daf1fbe4b5cde9f6bcc44a1a562c35b6d380a7f1bd97ecbb8cce5767db2'),
 ('Penduduk, Laju Pertumbuhan Penduduk, Distribusi Persentase Penduduk Kepadatan Penduduk, Rasio Jenis Kelamin Penduduk Menurut Kabupaten_Kota di Provinsi Kalimantan Utara, 2024.csv',
  '.csv',
  1426,
  '76859b29ed4ac2dda69045cd9a19c5d60633ba039b1e5f6d44114b5354a8eb14'),
 ('jumlah-penduduk-menurut-kabupaten-kota-di-provinsi-kalimantan-tengah-ribu-2020-2025(1).xlsx',
  '.xlsx',
  10470,
  '1d3b7f3f7e6f54c552e

## Later phases

1. Normalize administrative IDs and names.
2. Validate point-to-kabupaten attribution, including border cases.
3. Load existing grid-day outputs without changing score calculations.
4. Produce `kabupaten_day_summary`.
5. Prepare the dashboard payload and hierarchical UI behavior.
6. Run regression checks against the current province-level outputs.

## Phase 2 — Canonical administrative model

In [5]:
canonical_fields = ['province_code', 'province_name', 'kabupaten_code', 'kabupaten_name', 'geometry', '_source_file']
admin = kabupaten[canonical_fields].copy()
for column in ['province_code', 'province_name', 'kabupaten_code', 'kabupaten_name', '_source_file']:
    admin[column] = admin[column].astype(str).str.strip()
admin['admin_id'] = admin['kabupaten_code']
admin['province_code_from_kabupaten'] = admin['kabupaten_code'].str.split('.').str[0]
admin['province_consistent'] = admin['province_code'] == admin['province_code_from_kabupaten']
admin[['admin_id', 'province_code', 'province_name', 'kabupaten_name', 'province_consistent']].head()

,admin_id,province_code,province_name,kabupaten_name,province_consistent
0,61.07,61,Kalimantan Barat,Bengkayang,True
1,61.02,61,Kalimantan Barat,Mempawah,True
2,61.06,61,Kalimantan Barat,Kapuas Hulu,True
3,61.71,61,Kalimantan Barat,Kota Pontianak,True
4,61.12,61,Kalimantan Barat,Kubu Raya,True


In [6]:
expected_provinces = {'61', '62', '63', '64', '65'}
assert set(admin.province_code) == expected_provinces
assert admin.admin_id.is_unique
assert admin[['province_name', 'kabupaten_name']].notna().all().all()
assert admin.province_consistent.all()
assert admin.geometry.notna().all() and admin.geometry.is_valid.all()
assert len(admin) == 56
phase2_validation = pd.DataFrame([{'total_kabupaten_features': len(admin), 'unique_admin_ids': admin.admin_id.nunique(), 'province_count': admin.province_code.nunique(), 'invalid_geometries': int((~admin.geometry.is_valid).sum()), 'province_code_mismatches': int((~admin.province_consistent).sum()), 'duplicate_admin_ids': int(admin.admin_id.duplicated().sum())}])
phase2_validation

,total_kabupaten_features,unique_admin_ids,province_count,invalid_geometries,province_code_mismatches,duplicate_admin_ids
0,56,56,5,0,0,0


In [7]:
admin_lookup = (admin.drop(columns='geometry')[['admin_id', 'province_code', 'province_name', 'kabupaten_name', '_source_file']].rename(columns={'admin_id': 'kabupaten_code'}).sort_values('kabupaten_code').reset_index(drop=True))
admin_lookup.to_csv(OUTPUT / 'kabupaten_admin_lookup.csv', index=False)
admin_lookup

,kabupaten_code,province_code,province_name,kabupaten_name,_source_file
0,61.01,61,Kalimantan Barat,Sambas,Provinsi Kalimantan Barat-KAB_KOTA.geojson
1,61.02,61,Kalimantan Barat,Mempawah,Provinsi Kalimantan Barat-KAB_KOTA.geojson
2,61.03,61,Kalimantan Barat,Sanggau,Provinsi Kalimantan Barat-KAB_KOTA.geojson
3,61.04,61,Kalimantan Barat,Ketapang,Provinsi Kalimantan Barat-KAB_KOTA.geojson
4,61.05,61,Kalimantan Barat,Sintang,Provinsi Kalimantan Barat-KAB_KOTA.geojson
5,61.06,61,Kalimantan Barat,Kapuas Hulu,Provinsi Kalimantan Barat-KAB_KOTA.geojson
6,61.07,61,Kalimantan Barat,Bengkayang,Provinsi Kalimantan Barat-KAB_KOTA.geojson
7,61.08,61,Kalimantan Barat,Landak,Provinsi Kalimantan Barat-KAB_KOTA.geojson
8,61.09,61,Kalimantan Barat,Sekadau,Provinsi Kalimantan Barat-KAB_KOTA.geojson
9,61.10,61,Kalimantan Barat,Melawi,Provinsi Kalimantan Barat-KAB_KOTA.geojson


### Phase 2 result

The canonical model uses `kode_kk` as `kabupaten_code`, `kode_prov` as `province_code`, `kab_kota` as `kabupaten_name`, and `provinsi` as `province_name`. All 56 records pass uniqueness, geometry, and province-prefix checks. Population joins remain a later enrichment step.

## Phase 3 — Grid-day to kabupaten attribution

In [8]:
grid_path = OUTPUT / 'grid_day_priority.csv'
assert grid_path.exists(), f'Missing existing grid output: {grid_path}'
grid_days = pd.read_csv(grid_path)
required_grid_fields = {'grid_id', 'date', 'latitude', 'longitude', 'province_code', 'province_name', 'priority_score'}
missing_grid_fields = required_grid_fields - set(grid_days.columns)
assert not missing_grid_fields, f'Missing grid fields: {sorted(missing_grid_fields)}'
grid_points = gpd.GeoDataFrame(grid_days.copy(), geometry=gpd.points_from_xy(grid_days.longitude, grid_days.latitude), crs=4326)
grid_points.head()

,grid_id,date,detection_count,frp_median_mw,frp_max_mw,frp_missing,high_count,nominal_count,low_count,unknown_count,...,quality_note,activity_points,persistence_points,intensity_points,priority_score,priority_level,daily_rank,region_id,region_name,geometry
0,6:1101,2024-08-01,7,23.080,148.55,0,0,7,0,0,...,Jendela awal kurang dari 7 hari,39.809524,5.0,22.261905,67.0714,P2,1,0,Kalimantan Barat,POINT (110.13029 0.6437)
1,3:1104,2024-08-01,5,14.400,33.44,0,0,5,0,0,...,Jendela awal kurang dari 7 hari,38.285714,5.0,19.880952,63.1667,P2,2,0,Kalimantan Barat,POINT (110.43472 0.30698)
2,7:1097,2024-08-01,6,10.665,25.24,0,0,6,0,0,...,Jendela awal kurang dari 7 hari,39.238095,5.0,17.976190,62.2143,P2,3,0,Kalimantan Barat,POINT (109.75989 0.7637)
3,-3:1119,2024-08-01,4,14.065,25.34,0,0,4,0,0,...,Jendela awal kurang dari 7 hari,36.761905,5.0,19.642857,61.4048,P2,4,0,Kalimantan Barat,POINT (111.92082 -0.24846)
4,4:1105,2024-08-01,3,21.610,44.11,0,0,3,0,0,...,Jendela awal kurang dari 7 hari,33.904762,5.0,22.023810,60.9286,P2,5,0,Kalimantan Barat,POINT (110.57555 0.47567)


In [9]:
# Retain all candidate matches for the audit, then select one deterministic match.
candidates = gpd.sjoin(grid_points, admin[['admin_id', 'province_code', 'province_name', 'kabupaten_name', 'geometry']], how='left', predicate='intersects', lsuffix='grid', rsuffix='admin')
candidate_counts = candidates.groupby(candidates.index)['admin_id'].count()
attribution_ambiguous = candidate_counts.gt(1)
attribution_unmatched = candidate_counts.eq(0)
assigned = (candidates.sort_values(['admin_id'], na_position='last').loc[lambda frame: ~frame.index.duplicated(keep='first')].sort_index())
assigned['kabupaten_assignment_status'] = 'assigned'
assigned.loc[assigned.index.map(attribution_ambiguous).fillna(False), 'kabupaten_assignment_status'] = 'boundary_ambiguous'
assigned.loc[assigned.index.map(attribution_unmatched).fillna(False), 'kabupaten_assignment_status'] = 'unmatched'
assigned['kabupaten_code'] = assigned['admin_id'].fillna('')
assigned['kabupaten_name'] = assigned['kabupaten_name'].fillna('Kabupaten/kota belum terpetakan')
assigned[['grid_id', 'date', 'province_name_grid', 'kabupaten_code', 'kabupaten_name', 'kabupaten_assignment_status']].head()

,grid_id,date,province_name_grid,kabupaten_code,kabupaten_name,kabupaten_assignment_status
0,6:1101,2024-08-01,Kalimantan Barat,61.08,Landak,assigned
1,3:1104,2024-08-01,Kalimantan Barat,61.03,Sanggau,assigned
2,7:1097,2024-08-01,Kalimantan Barat,61.08,Landak,assigned
3,-3:1119,2024-08-01,Kalimantan Barat,61.10,Melawi,assigned
4,4:1105,2024-08-01,Kalimantan Barat,61.03,Sanggau,assigned


In [10]:
# The join must not alter the existing score or row grain.
assert len(assigned) == len(grid_days)
assert assigned.priority_score.reset_index(drop=True).equals(grid_days.priority_score.reset_index(drop=True))
assert assigned[['grid_id', 'date']].duplicated().sum() == 0
assert set(assigned.kabupaten_assignment_status) <= {'assigned', 'boundary_ambiguous', 'unmatched'}
phase3_audit = (assigned.groupby('kabupaten_assignment_status', dropna=False).size().rename('grid_day_rows').reset_index())
phase3_audit

,kabupaten_assignment_status,grid_day_rows
0,assigned,26798


In [11]:
assigned_export = assigned.rename(columns={'province_code_grid': 'province_code', 'province_name_grid': 'province_name'})
attribution_columns = [c for c in grid_days.columns if c in assigned_export.columns]
attributed_grid_days = assigned_export[attribution_columns + ['kabupaten_code', 'kabupaten_name', 'kabupaten_assignment_status']].copy()
attributed_grid_days.to_csv(OUTPUT / 'grid_day_kabupaten_attribution.csv', index=False)
phase3_summary = pd.DataFrame([{
    'grid_day_rows': len(attributed_grid_days),
    'assigned_rows': int((assigned.kabupaten_assignment_status == 'assigned').sum()),
    'boundary_ambiguous_rows': int((assigned.kabupaten_assignment_status == 'boundary_ambiguous').sum()),
    'unmatched_rows': int((assigned.kabupaten_assignment_status == 'unmatched').sum()),
    'unique_kabupaten_assigned': int(assigned.loc[assigned.kabupaten_code.ne(''), 'kabupaten_code'].nunique()),
}])
phase3_summary

,grid_day_rows,assigned_rows,boundary_ambiguous_rows,unmatched_rows,unique_kabupaten_assigned
0,26798,26798,0,0,56


### Phase 3 result

The attribution output preserves the existing grid-day row grain and priority scores. Each row carries a canonical kabupaten code/name plus an explicit assignment status. Ambiguous or unmatched points remain auditable rather than being silently duplicated or dropped.

## Phase 4 — Kabupaten/day analytical summaries

In [12]:
summary_input = attributed_grid_days.copy()
summary_input['last_seen_utc'] = pd.to_datetime(summary_input['last_seen_utc'], utc=True, errors='coerce')
summary_input['assigned_to_kabupaten'] = summary_input['kabupaten_code'].ne('')
summary_input['valid_frp_median'] = pd.to_numeric(summary_input['frp_median_mw'], errors='coerce')
summary_input['p1_grid_count'] = summary_input['priority_level'].eq('P1').astype(int)
summary_input['p2_grid_count'] = summary_input['priority_level'].eq('P2').astype(int)
summary_input['p3_grid_count'] = summary_input['priority_level'].eq('P3').astype(int)
summary_input.head()

,grid_id,date,detection_count,frp_median_mw,frp_max_mw,frp_missing,high_count,nominal_count,low_count,unknown_count,...,region_id,region_name,kabupaten_code,kabupaten_name,kabupaten_assignment_status,assigned_to_kabupaten,valid_frp_median,p1_grid_count,p2_grid_count,p3_grid_count
0,6:1101,2024-08-01,7,23.080,148.55,0,0,7,0,0,...,0,Kalimantan Barat,61.08,Landak,assigned,True,23.080,0,1,0
1,3:1104,2024-08-01,5,14.400,33.44,0,0,5,0,0,...,0,Kalimantan Barat,61.03,Sanggau,assigned,True,14.400,0,1,0
2,7:1097,2024-08-01,6,10.665,25.24,0,0,6,0,0,...,0,Kalimantan Barat,61.08,Landak,assigned,True,10.665,0,1,0
3,-3:1119,2024-08-01,4,14.065,25.34,0,0,4,0,0,...,0,Kalimantan Barat,61.10,Melawi,assigned,True,14.065,0,1,0
4,4:1105,2024-08-01,3,21.610,44.11,0,0,3,0,0,...,0,Kalimantan Barat,61.03,Sanggau,assigned,True,21.610,0,1,0


In [13]:
kabupaten_day_summary = (summary_input.loc[summary_input.assigned_to_kabupaten]
    .groupby(['date', 'province_code', 'province_name', 'kabupaten_code', 'kabupaten_name'], as_index=False)
    .agg(
        active_grid_count=('grid_id', 'nunique'),
        detection_count=('detection_count', 'sum'),
        p1_grid_count=('p1_grid_count', 'sum'),
        p2_grid_count=('p2_grid_count', 'sum'),
        p3_grid_count=('p3_grid_count', 'sum'),
        highest_priority_score=('priority_score', 'max'),
        average_priority_score=('priority_score', 'mean'),
        median_frp_mw=('valid_frp_median', 'median'),
        latest_observation_utc=('last_seen_utc', 'max'),
        active_days_7d=('active_days_7d', 'max'),
    )
    .sort_values(['date', 'highest_priority_score', 'kabupaten_code'], ascending=[True, False, True])
    .reset_index(drop=True))
kabupaten_day_summary[['highest_priority_score', 'average_priority_score', 'median_frp_mw']] = kabupaten_day_summary[['highest_priority_score', 'average_priority_score', 'median_frp_mw']].round(4)
kabupaten_day_summary.head()

,date,province_code,province_name,kabupaten_code,kabupaten_name,active_grid_count,detection_count,p1_grid_count,p2_grid_count,p3_grid_count,highest_priority_score,average_priority_score,median_frp_mw,latest_observation_utc,active_days_7d
0,2024-08-01,61,Kalimantan Barat,61.08,Landak,6,21,0,2,4,67.0714,50.2778,10.0275,2024-08-01 05:46:00+00:00,1
1,2024-08-01,61,Kalimantan Barat,61.03,Sanggau,40,88,0,2,38,63.1667,43.8577,8.5325,2024-08-01 05:46:00+00:00,1
2,2024-08-01,61,Kalimantan Barat,61.10,Melawi,1,4,0,1,0,61.4048,61.4048,14.0650,2024-08-01 05:46:00+00:00,1
3,2024-08-01,61,Kalimantan Barat,61.09,Sekadau,8,14,0,0,8,59.2619,44.1072,15.6600,2024-08-01 05:46:00+00:00,1
4,2024-08-01,61,Kalimantan Barat,61.04,Ketapang,4,6,0,0,4,57.2619,38.9048,17.5450,2024-08-01 05:45:00+00:00,1


In [14]:
# Summary-level invariants and reconciliation checks.
assert kabupaten_day_summary[['date', 'kabupaten_code']].duplicated().sum() == 0
assert (kabupaten_day_summary['detection_count'] > 0).all()
assert (kabupaten_day_summary['active_grid_count'] > 0).all()
assert (kabupaten_day_summary['p1_grid_count'] + kabupaten_day_summary['p2_grid_count'] + kabupaten_day_summary['p3_grid_count'] == kabupaten_day_summary['active_grid_count']).all()
daily_grid_counts = summary_input.loc[summary_input.assigned_to_kabupaten].groupby('date').size()
daily_summary_counts = kabupaten_day_summary.groupby('date').active_grid_count.sum()
assert daily_grid_counts.equals(daily_summary_counts)
phase4_validation = pd.DataFrame([{
    'kabupaten_day_rows': len(kabupaten_day_summary),
    'unique_kabupaten': kabupaten_day_summary.kabupaten_code.nunique(),
    'date_start': kabupaten_day_summary.date.min(),
    'date_end': kabupaten_day_summary.date.max(),
    'max_daily_kabupaten_count': int(kabupaten_day_summary.groupby('date').size().max()),
}])
phase4_validation

,kabupaten_day_rows,unique_kabupaten,date_start,date_end,max_daily_kabupaten_count
0,7086,56,2024-08-01,2026-05-31,46


In [15]:
kabupaten_day_summary['latest_observation_utc'] = kabupaten_day_summary['latest_observation_utc'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
kabupaten_day_summary.to_csv(OUTPUT / 'kabupaten_day_summary.csv', index=False)
kabupaten_day_summary.to_json(OUTPUT / 'kabupaten_day_summary.json', orient='records', force_ascii=False, indent=2)
kabupaten_day_summary.head(10)

,date,province_code,province_name,kabupaten_code,kabupaten_name,active_grid_count,detection_count,p1_grid_count,p2_grid_count,p3_grid_count,highest_priority_score,average_priority_score,median_frp_mw,latest_observation_utc,active_days_7d
0,2024-08-01,61,Kalimantan Barat,61.08,Landak,6,21,0,2,4,67.0714,50.2778,10.0275,2024-08-01T05:46:00Z,1
1,2024-08-01,61,Kalimantan Barat,61.03,Sanggau,40,88,0,2,38,63.1667,43.8577,8.5325,2024-08-01T05:46:00Z,1
2,2024-08-01,61,Kalimantan Barat,61.10,Melawi,1,4,0,1,0,61.4048,61.4048,14.0650,2024-08-01T05:46:00Z,1
3,2024-08-01,61,Kalimantan Barat,61.09,Sekadau,8,14,0,0,8,59.2619,44.1072,15.6600,2024-08-01T05:46:00Z,1
4,2024-08-01,61,Kalimantan Barat,61.04,Ketapang,4,6,0,0,4,57.2619,38.9048,17.5450,2024-08-01T05:45:00Z,1
5,2024-08-01,61,Kalimantan Barat,61.07,Bengkayang,2,3,0,0,2,56.0714,42.9523,19.8575,2024-08-01T05:46:00Z,1
6,2024-08-01,62,Kalimantan Tengah,62.05,Barito Utara,1,2,0,0,1,48.2143,48.2143,8.5800,2024-08-01T05:45:00Z,1
7,2024-08-01,64,Kalimantan Timur,64.02,Kutai Kartanegara,4,7,0,0,4,42.3571,27.4881,3.6450,2024-08-01T18:27:00Z,1
8,2024-08-01,63,Kalimantan Selatan,63.11,Balangan,1,3,0,0,1,41.4048,41.4048,2.6500,2024-08-01T18:27:00Z,1
9,2024-08-01,61,Kalimantan Barat,61.06,Kapuas Hulu,5,6,0,0,5,40.3571,26.9857,3.4900,2024-08-01T05:46:00Z,1


### Phase 4 result

The summary table is descriptive: it aggregates existing grid-day scores and does not create a new kabupaten risk score. Population fields will be joined later as contextual metadata.

## Phase 5 — Population context enrichment

In [16]:
ADMIN_NAME_ALIASES = {
    'kota baru': 'kotabaru',
    'banjar baru': 'banjarbaru',
    'palangka raya': 'palangkaraya',
    'kotawaringintimur': 'kotawaringin timur',
}

def clean_admin_name(value):
    value = str(value).strip()
    value = value.replace('Kabupaten ', '').replace('Kab. ', '')
    value = ' '.join(value.split()).casefold()
    value = value.removeprefix('kota ')
    value = ADMIN_NAME_ALIASES.get(value, value)
    return value.replace(' ', '')

population_parts = []

# Kalimantan Barat uses a generic filename but contains kabupaten/kota rows.
kalbar_path = next(p for p in ADDITIONS.glob('*.csv') if p.name.startswith('Jumlah Penduduk Menurut') and 'Provinsi' not in p.name)
kalbar_raw = pd.read_csv(kalbar_path, header=None, encoding='utf-8-sig')
kalbar = kalbar_raw.iloc[3:, :2].copy()
kalbar.columns = ['source_name', 'population']
kalbar['province_code'] = '61'
kalbar['population_density'] = pd.NA
kalbar['population_year'] = 2024
kalbar['population_thousand'] = pd.to_numeric(kalbar['population'], errors='coerce') / 1000
kalbar = kalbar.loc[kalbar.source_name.ne('Kalimantan Barat')].copy()
population_parts.append(kalbar[['province_code', 'source_name', 'population_thousand', 'population_density', 'population_year']])

# Kalimantan Selatan and Kalimantan Utara use the same six-column CSV layout.
for path, province_code in [(next(ADDITIONS.glob('*Kalimantan Selatan*.csv')), '63'), (next(ADDITIONS.glob('*Kalimantan Utara*.csv')), '65')]:
    frame = pd.read_csv(path, encoding='utf-8-sig')
    frame = frame.rename(columns={frame.columns[0]: 'source_name', frame.columns[1]: 'population_thousand', frame.columns[4]: 'population_density'})
    frame['province_code'] = province_code
    frame['population_year'] = 2024
    population_parts.append(frame[['province_code', 'source_name', 'population_thousand', 'population_density', 'population_year']])

# Kalimantan Timur has a two-row title/header layout and population in thousands.
kaltim_path = next(ADDITIONS.glob('*Kalimantan Timur*.csv'))
kaltim_raw = pd.read_csv(kaltim_path, header=None, encoding='utf-8-sig')
kaltim = kaltim_raw.iloc[3:, :2].copy()
kaltim.columns = ['source_name', 'population_thousand']
kaltim['province_code'] = '64'
kaltim['population_density'] = pd.NA
kaltim['population_year'] = 2024
population_parts.append(kaltim[['province_code', 'source_name', 'population_thousand', 'population_density', 'population_year']])

# Kalimantan Tengah has a three-row title/header layout; use the 2024 column.
kalteng_path = next(ADDITIONS.glob('*kalimantan-tengah*.xlsx'))
kalteng_raw = pd.read_excel(kalteng_path, header=None)
kalteng = kalteng_raw.iloc[5:, [1, 3]].copy()
kalteng.columns = ['source_name', 'population_thousand']
kalteng['province_code'] = '62'
kalteng['population_density'] = pd.NA
kalteng['population_year'] = 2024
population_parts.append(kalteng[['province_code', 'source_name', 'population_thousand', 'population_density', 'population_year']])

population_raw = pd.concat(population_parts, ignore_index=True)
population_raw['source_name'] = population_raw['source_name'].astype(str).str.strip()
population_raw['population_thousand'] = pd.to_numeric(population_raw['population_thousand'], errors='coerce')
population_raw['population'] = population_raw['population_thousand'] * 1000
population_raw['population_name_key'] = population_raw['source_name'].map(clean_admin_name)
population_raw = population_raw.loc[population_raw.population.notna() & population_raw.population_name_key.ne('nan')].copy()
population_raw.loc[(population_raw.province_code == '63') & population_raw.source_name.str.casefold().eq('kota baru'), 'population_name_key'] = 'kotabaru'
population_raw[['province_code', 'source_name', 'population', 'population_density', 'population_year']]

C:\Users\Inaka\AppData\Local\Temp\ipykernel_22272\137224777.py:58: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  population_raw = pd.concat(population_parts, ignore_index=True)


,province_code,source_name,population,population_density,population_year
0,61,Sambas,654912.0,NaN,2024
1,61,Bengkayang,303422.0,NaN,2024
2,61,Landak,419092.0,NaN,2024
3,61,Mempawah,316728.0,NaN,2024
4,61,Sanggau,510412.0,NaN,2024
5,61,Ketapang,599890.0,NaN,2024
6,61,Sintang,443684.0,NaN,2024
7,61,Kapuas Hulu,265772.0,NaN,2024
8,61,Sekadau,222510.0,NaN,2024
9,61,Melawi,243250.0,NaN,2024


In [17]:
admin_lookup['population_name_key'] = admin_lookup['kabupaten_name'].map(clean_admin_name)
population_join = admin_lookup.merge(population_raw, on=['province_code', 'population_name_key'], how='left', indicator=True)
population_join['population_source'] = population_join['_source_file'].where(population_join['_merge'].eq('both'), pd.NA)
population_coverage = (population_join.groupby('province_code', as_index=False)
    .agg(kabupaten_count=('kabupaten_code', 'size'), matched_count=('population', lambda s: int(s.notna().sum())))
    .assign(unmatched_count=lambda d: d.kabupaten_count - d.matched_count))
population_coverage

,province_code,kabupaten_count,matched_count,unmatched_count
0,61,14,14,0
1,62,14,14,0
2,63,13,13,0
3,64,10,10,0
4,65,5,5,0


In [18]:
# Population is context only and is not joined into the priority score.
assert population_join.kabupaten_code.is_unique
assert set(population_join.province_code) == {'61', '62', '63', '64', '65'}
population_join[['kabupaten_code', 'province_code', 'kabupaten_name', 'population', 'population_density', 'population_year', '_merge']].to_csv(OUTPUT / 'kabupaten_population_context.csv', index=False)
population_join[['kabupaten_code', 'province_code', 'kabupaten_name', 'population', 'population_density', 'population_year', '_merge']].head()

,kabupaten_code,province_code,kabupaten_name,population,population_density,population_year,_merge
0,61.01,61,Sambas,654912.0,NaN,2024,both
1,61.02,61,Mempawah,316728.0,NaN,2024,both
2,61.03,61,Sanggau,510412.0,NaN,2024,both
3,61.04,61,Ketapang,599890.0,NaN,2024,both
4,61.05,61,Sintang,443684.0,NaN,2024,both


### Phase 5 result

Population sources are normalized into people, with source units retained in the notebook logic. The province-total CSV is excluded from kabupaten joins. Any unmatched kabupaten remains explicitly missing and population does not affect priority scoring.

## Phase 6 — Enriched kabupaten/day output for dashboard consumption

In [19]:
population_context = population_join[['kabupaten_code', 'population', 'population_density', 'population_year']].copy()
kabupaten_day_enriched = kabupaten_day_summary.merge(population_context, on='kabupaten_code', how='left', validate='many_to_one')
kabupaten_day_enriched['population_context_note'] = 'Population context; not included in priority score.'
kabupaten_day_enriched.head()

,date,province_code,province_name,kabupaten_code,kabupaten_name,active_grid_count,detection_count,p1_grid_count,p2_grid_count,p3_grid_count,highest_priority_score,average_priority_score,median_frp_mw,latest_observation_utc,active_days_7d,population,population_density,population_year,population_context_note
0,2024-08-01,61,Kalimantan Barat,61.08,Landak,6,21,0,2,4,67.0714,50.2778,10.0275,2024-08-01T05:46:00Z,1,419092.0,NaN,2024,Population context; not included in priority s...
1,2024-08-01,61,Kalimantan Barat,61.03,Sanggau,40,88,0,2,38,63.1667,43.8577,8.5325,2024-08-01T05:46:00Z,1,510412.0,NaN,2024,Population context; not included in priority s...
2,2024-08-01,61,Kalimantan Barat,61.10,Melawi,1,4,0,1,0,61.4048,61.4048,14.0650,2024-08-01T05:46:00Z,1,243250.0,NaN,2024,Population context; not included in priority s...
3,2024-08-01,61,Kalimantan Barat,61.09,Sekadau,8,14,0,0,8,59.2619,44.1072,15.6600,2024-08-01T05:46:00Z,1,222510.0,NaN,2024,Population context; not included in priority s...
4,2024-08-01,61,Kalimantan Barat,61.04,Ketapang,4,6,0,0,4,57.2619,38.9048,17.5450,2024-08-01T05:45:00Z,1,599890.0,NaN,2024,Population context; not included in priority s...


In [20]:
assert len(kabupaten_day_enriched) == len(kabupaten_day_summary)
assert kabupaten_day_enriched[['kabupaten_code', 'population']].drop_duplicates().kabupaten_code.is_unique
assert kabupaten_day_enriched.population.notna().all()
population_by_admin = population_context.set_index('kabupaten_code').population
assert kabupaten_day_enriched.apply(lambda row: row.population == population_by_admin[row.kabupaten_code], axis=1).all()
phase6_validation = pd.DataFrame([{'summary_rows': len(kabupaten_day_enriched), 'unique_kabupaten': kabupaten_day_enriched.kabupaten_code.nunique(), 'population_join_coverage': float(kabupaten_day_enriched.population.notna().mean()), 'density_available_coverage': float(kabupaten_day_enriched.population_density.notna().mean())}])
phase6_validation

,summary_rows,unique_kabupaten,population_join_coverage,density_available_coverage
0,7086,56,1.0,0.236946


In [21]:
kabupaten_day_enriched.to_csv(OUTPUT / 'kabupaten_day_enriched.csv', index=False)
kabupaten_day_enriched.to_json(OUTPUT / 'kabupaten_day_enriched.json', orient='records', force_ascii=False, indent=2)
kabupaten_day_enriched.head(10)

,date,province_code,province_name,kabupaten_code,kabupaten_name,active_grid_count,detection_count,p1_grid_count,p2_grid_count,p3_grid_count,highest_priority_score,average_priority_score,median_frp_mw,latest_observation_utc,active_days_7d,population,population_density,population_year,population_context_note
0,2024-08-01,61,Kalimantan Barat,61.08,Landak,6,21,0,2,4,67.0714,50.2778,10.0275,2024-08-01T05:46:00Z,1,419092.0,NaN,2024,Population context; not included in priority s...
1,2024-08-01,61,Kalimantan Barat,61.03,Sanggau,40,88,0,2,38,63.1667,43.8577,8.5325,2024-08-01T05:46:00Z,1,510412.0,NaN,2024,Population context; not included in priority s...
2,2024-08-01,61,Kalimantan Barat,61.10,Melawi,1,4,0,1,0,61.4048,61.4048,14.0650,2024-08-01T05:46:00Z,1,243250.0,NaN,2024,Population context; not included in priority s...
3,2024-08-01,61,Kalimantan Barat,61.09,Sekadau,8,14,0,0,8,59.2619,44.1072,15.6600,2024-08-01T05:46:00Z,1,222510.0,NaN,2024,Population context; not included in priority s...
4,2024-08-01,61,Kalimantan Barat,61.04,Ketapang,4,6,0,0,4,57.2619,38.9048,17.5450,2024-08-01T05:45:00Z,1,599890.0,NaN,2024,Population context; not included in priority s...
5,2024-08-01,61,Kalimantan Barat,61.07,Bengkayang,2,3,0,0,2,56.0714,42.9523,19.8575,2024-08-01T05:46:00Z,1,303422.0,NaN,2024,Population context; not included in priority s...
6,2024-08-01,62,Kalimantan Tengah,62.05,Barito Utara,1,2,0,0,1,48.2143,48.2143,8.5800,2024-08-01T05:45:00Z,1,162620.0,NaN,2024,Population context; not included in priority s...
7,2024-08-01,64,Kalimantan Timur,64.02,Kutai Kartanegara,4,7,0,0,4,42.3571,27.4881,3.6450,2024-08-01T18:27:00Z,1,789770.0,NaN,2024,Population context; not included in priority s...
8,2024-08-01,63,Kalimantan Selatan,63.11,Balangan,1,3,0,0,1,41.4048,41.4048,2.6500,2024-08-01T18:27:00Z,1,138100.0,76.0,2024,Population context; not included in priority s...
9,2024-08-01,61,Kalimantan Barat,61.06,Kapuas Hulu,5,6,0,0,5,40.3571,26.9857,3.4900,2024-08-01T05:46:00Z,1,265772.0,NaN,2024,Population context; not included in priority s...


### Phase 6 result

The dashboard-ready kabupaten/day table now combines administrative attribution, transparent grid-level score aggregates, and population context. Population remains descriptive metadata and does not affect ranking or priority levels.

## Phase 7 — Dashboard payload preparation

In [22]:
display_admin = admin.copy()
display_admin['geometry'] = display_admin.geometry.simplify(0.003, preserve_topology=True)
display_admin = display_admin[['admin_id', 'province_code', 'province_name', 'kabupaten_name', 'geometry']].rename(columns={'admin_id': 'kabupaten_code'})
kabupaten_boundaries_geojson = json.loads(display_admin.to_json(drop_id=True))
len(kabupaten_boundaries_geojson['features'])

56

In [23]:
grid_payload_columns = ['grid_id', 'date', 'latitude', 'longitude', 'province_code', 'province_name', 'kabupaten_code', 'kabupaten_name', 'kabupaten_assignment_status', 'detection_count', 'active_days_7d', 'detections_7d', 'frp_median_mw', 'frp_max_mw', 'priority_score', 'priority_level', 'activity_points', 'persistence_points', 'intensity_points', 'last_seen_utc', 'nearest_school_km', 'nearest_school_name', 'quality_note']
grid_payload = attributed_grid_days[grid_payload_columns].copy()
grid_payload['last_seen_utc'] = pd.to_datetime(grid_payload['last_seen_utc'], utc=True, errors='coerce').dt.strftime('%Y-%m-%dT%H:%M:%SZ')
grid_payload_rows = grid_payload.where(pd.notna(grid_payload), None).to_dict(orient='records')
summary_payload_rows = kabupaten_day_enriched.where(pd.notna(kabupaten_day_enriched), None).to_dict(orient='records')
len(grid_payload_rows), len(summary_payload_rows)

(26798, 7086)

In [24]:
dashboard_payload = {
    'contract_version': 'kabupaten-drilldown-1.0',
    'generated_from': '07_FIRELINE_Kabupaten_Drilldown.ipynb',
    'score_semantics': 'Grid-day scores remain calculated against all active Kalimantan areas on the same date.',
    'population_semantics': 'Population is contextual metadata and is not included in priority scoring.',
    'boundaries': kabupaten_boundaries_geojson,
    'kabupaten_day_summary': summary_payload_rows,
    'grid_day_attribution': grid_payload_rows,
}
payload_path = OUTPUT / 'kabupaten_dashboard_payload.json'
payload_path.write_text(json.dumps(dashboard_payload, ensure_ascii=False, default=str), encoding='utf-8')
payload_path.stat().st_size

23727679

In [25]:
# Payload contract checks.
payload_check = json.loads(payload_path.read_text(encoding='utf-8'))
assert payload_check['contract_version'] == 'kabupaten-drilldown-1.0'
assert len(payload_check['boundaries']['features']) == 56
assert len(payload_check['grid_day_attribution']) == len(attributed_grid_days)
assert len(payload_check['kabupaten_day_summary']) == len(kabupaten_day_enriched)
assert payload_check['population_semantics'].startswith('Population is contextual')
pd.DataFrame([{'boundary_features': len(payload_check['boundaries']['features']), 'grid_rows': len(payload_check['grid_day_attribution']), 'summary_rows': len(payload_check['kabupaten_day_summary']), 'payload_bytes': payload_path.stat().st_size}])

,boundary_features,grid_rows,summary_rows,payload_bytes
0,56,26798,7086,23727679


### Phase 7 result

A versioned dashboard payload now contains the kabupaten boundary layer, enriched kabupaten/day summaries, and grid-level attribution. The current UI has not been changed yet; this payload is the contract for the next interface phase.

## Phase 8 — Hierarchical dashboard filter contract

In [26]:
payload_grid = pd.DataFrame(payload_check['grid_day_attribution'])
payload_summary = pd.DataFrame(payload_check['kabupaten_day_summary'])
payload_grid['date'] = payload_grid['date'].astype(str)
payload_summary['date'] = payload_summary['date'].astype(str)
available_dates = sorted(payload_grid['date'].dropna().unique().tolist())
available_provinces = (payload_grid[['province_code', 'province_name']].drop_duplicates().sort_values('province_code').to_dict(orient='records'))
available_kabupaten = (payload_grid[['kabupaten_code', 'kabupaten_name', 'province_code', 'province_name']].drop_duplicates().loc[lambda d: d.kabupaten_code.ne('')].sort_values(['province_code', 'kabupaten_code']).to_dict(orient='records'))
len(available_dates), len(available_provinces), len(available_kabupaten)

(624, 5, 56)

In [27]:
def filter_scope(date=None, province_code=None, kabupaten_code=None, priority_level=None):
    frame = payload_grid.copy()
    if date: frame = frame.loc[frame.date.eq(date)]
    if province_code: frame = frame.loc[frame.province_code.astype(str).eq(str(province_code))]
    if kabupaten_code: frame = frame.loc[frame.kabupaten_code.eq(kabupaten_code)]
    if priority_level: frame = frame.loc[frame.priority_level.eq(priority_level)]
    return frame

demo_date = available_dates[len(available_dates) // 2]
province_demo = available_provinces[0]['province_code']
kabupaten_demo = next(row['kabupaten_code'] for row in available_kabupaten if row['province_code'] == province_demo)
scope_checks = pd.DataFrame([{
    'all_scope_rows': len(filter_scope(date=demo_date)),
    'province_scope_rows': len(filter_scope(date=demo_date, province_code=province_demo)),
    'kabupaten_scope_rows': len(filter_scope(date=demo_date, province_code=province_demo, kabupaten_code=kabupaten_demo)),
    'nested_scope_is_subset': set(filter_scope(date=demo_date, province_code=province_demo, kabupaten_code=kabupaten_demo).grid_id).issubset(set(filter_scope(date=demo_date, province_code=province_demo).grid_id)),
}])
scope_checks

,all_scope_rows,province_scope_rows,kabupaten_scope_rows,nested_scope_is_subset
0,32,12,1,True


In [28]:
filter_contract = {
    'controls': ['date', 'province_code', 'kabupaten_code', 'priority_level'],
    'cascade': {'province_code': 'filters kabupaten_code options', 'kabupaten_code': 'filters grid rows'},
    'reset': {'date': 'default demo date', 'province_code': '', 'kabupaten_code': '', 'priority_level': ''},
    'selection': {'polygon_click': 'sets kabupaten_code', 'marker_click': 'opens grid detail', 'clear_kabupaten': 'returns to province scope'},
    'empty_state': 'No data is not interpreted as safe or risk-free.',
    'score_rule': 'Filtering does not recalculate priority percentiles.',
}
(OUTPUT / 'dashboard_filter_contract.json').write_text(json.dumps(filter_contract, ensure_ascii=False, indent=2), encoding='utf-8')
filter_contract

{'controls': ['date', 'province_code', 'kabupaten_code', 'priority_level'],
 'cascade': {'province_code': 'filters kabupaten_code options',
  'kabupaten_code': 'filters grid rows'},
 'reset': {'date': 'default demo date',
  'province_code': '',
  'kabupaten_code': '',
  'priority_level': ''},
 'selection': {'polygon_click': 'sets kabupaten_code',
  'marker_click': 'opens grid detail',
  'clear_kabupaten': 'returns to province scope'},
 'empty_state': 'No data is not interpreted as safe or risk-free.',
 'score_rule': 'Filtering does not recalculate priority percentiles.'}

### Phase 8 result

The cascading filter contract is validated against the real payload. The live HTML dashboard has not yet been changed; the next implementation phase can use this contract to add controls without changing score semantics.

## Phase 10 — Render the canonical kabupaten dashboard artifact

In [29]:
# Adapt the canonical payload to the existing portable dashboard template.
template = (ROOT / 'web' / 'dashboard.html').read_text(encoding='utf-8')
audit_summary_path = OUTPUT / 'audit_summary.json'
audit_summary = json.loads(audit_summary_path.read_text(encoding='utf-8')) if audit_summary_path.exists() else {}
dashboard_columns = ['grid_id', 'date', 'latitude', 'longitude', 'province_name', 'kabupaten_code', 'kabupaten_name', 'detection_count', 'active_days_7d', 'detections_7d', 'frp_median_mw', 'frp_max_mw', 'priority_score', 'priority_level', 'activity_points', 'persistence_points', 'intensity_points', 'high_count', 'nominal_count', 'low_count', 'unknown_count', 'night_count', 'nearest_school_km', 'nearest_school_name', 'last_seen_utc', 'quality_note', 'grid_x', 'grid_y']
dashboard_rows_frame = pd.DataFrame(payload_check['grid_day_attribution']).copy()
dashboard_rows_frame['region_id'] = dashboard_rows_frame['kabupaten_code']
dashboard_rows_frame['region_name'] = dashboard_rows_frame['kabupaten_name']
dashboard_columns = ['grid_id', 'date', 'latitude', 'longitude', 'province_name', 'region_id', 'region_name', 'detection_count', 'active_days_7d', 'detections_7d', 'frp_median_mw', 'frp_max_mw', 'priority_score', 'priority_level', 'activity_points', 'persistence_points', 'intensity_points', 'high_count', 'nominal_count', 'low_count', 'unknown_count', 'night_count', 'nearest_school_km', 'nearest_school_name', 'last_seen_utc', 'quality_note', 'grid_x', 'grid_y']
for col in ['grid_x', 'grid_y', 'high_count', 'nominal_count', 'low_count', 'unknown_count', 'night_count']:
    if col not in dashboard_rows_frame: dashboard_rows_frame[col] = None
dashboard_rows_frame['last_seen_utc'] = pd.to_datetime(dashboard_rows_frame['last_seen_utc'], utc=True, errors='coerce').dt.strftime('%Y-%m-%dT%H:%M:%SZ')
dashboard_payload_compat = {'columns': dashboard_columns, 'rows': dashboard_rows_frame[dashboard_columns].where(pd.notna(dashboard_rows_frame[dashboard_columns]), None).values.tolist(), 'summary': audit_summary, 'boundaries': payload_check['boundaries']}
dashboard_payload_compat['summary']['region_note'] = 'Kabupaten/kota boundaries from data/additions; polygons simplified for display.'
encoded_dashboard = json.dumps(dashboard_payload_compat, ensure_ascii=False, separators=(',', ':'), default=str).replace('<', '\\u003c')
dashboard_html = template.replace('__FIRELINE_DATA__', encoded_dashboard)
dashboard_output = OUTPUT / 'FIRELINE_Priority_Map.html'
dashboard_output.write_text(dashboard_html, encoding='utf-8')
dashboard_output.stat().st_size

8780668

In [30]:
artifact_text = dashboard_output.read_text(encoding='utf-8')
assert 'kabupaten_code' in artifact_text
assert 'Kabupaten/kota' in artifact_text
assert 'id="fireline-data"' in artifact_text
pd.DataFrame([{'artifact': str(dashboard_output), 'bytes': dashboard_output.stat().st_size, 'kabupaten_boundary_features': len(payload_check['boundaries']['features']), 'grid_rows': len(dashboard_rows_frame)}])

,artifact,bytes,kabupaten_boundary_features,grid_rows
0,C:\Users\Inaka\Downloads\Fireline-checkpoint3\...,8780668,56,26798


### Phase 10 result

The portable dashboard artifact is now generated from the canonical kabupaten payload. The dashboard retains the existing visual/detail contract while receiving stable kabupaten IDs and the new boundary layer.